# Notebook 6 - Transformer enrichi

Ce notebook est une experience d'amelioration apres le projet initial. Il reprend l'idee du notebook 4, mais corrige sa principale limite : le Transformer n'etait entraine que sur un tres petit echantillon.

Objectif : fine-tuner un Transformer sur la taxonomie enrichie du notebook 5, avec davantage de donnees et une evaluation comparable sur le domaine NovelForge.

## Positionnement

Les notebooks 1 a 4 constituent le projet initial : EDA, baseline ML, LSTM et Transformer de demonstration.

Les notebooks 5 et 6 sont des experiences d'amelioration :

- notebook 5 : meilleur travail sur les donnees, taxonomie regroupee, baseline enrichie ;
- notebook 6 : tentative de Transformer plus serieux avec le dataset enrichi.

Le Transformer est le modele le plus puissant en theorie, mais il a besoin de plus de donnees et de fine-tuning pour exprimer cet avantage.

In [ ]:
from pathlib import Path
import os
import sys
import time

# Works whether Jupyter starts from the project root or from notebooks/.
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / 'src').exists() and (PROJECT_DIR.parent / 'src').exists():
    PROJECT_DIR = PROJECT_DIR.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import classification_report, f1_score, hamming_loss, jaccard_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

from src.baseline_ml import apply_thresholds, find_best_global_threshold, find_best_label_thresholds
from src.enriched_dataset import build_enriched_dataset, load_original_dataset, summarize_labels
from src.project_config import ENRICHED_GENRE_VOCABULARY
from src.transformer_model import NovelForgeTransformer, TransformerConfig

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)

DEVICE_KIND = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device available: {DEVICE_KIND}')
PROJECT_DIR

## Parametres ajustables

Les valeurs ci-dessous sont un compromis. Sur CPU, elles restent plus lourdes que le notebook 4 mais encore raisonnables. Sur GPU, on peut augmenter les lignes et le nombre d'epochs.

In [ ]:
if DEVICE_KIND == 'cuda':
    CURRENT_TRAIN_ROWS = 8_000
    MAL_TRAIN_ROWS = 12_000
    VALID_ROWS = 2_000
    TEST_ROWS = 3_000
    EPOCHS = 3
    TRAIN_BATCH_SIZE = 16
    EVAL_BATCH_SIZE = 32
else:
    CURRENT_TRAIN_ROWS = 2_000
    MAL_TRAIN_ROWS = 3_000
    VALID_ROWS = 800
    TEST_ROWS = 1_200
    EPOCHS = 2
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16

MAX_LENGTH = 192
MODEL_NAME = 'distilbert-base-uncased'
RANDOM_STATE = 42

print({
    'current_train_rows': CURRENT_TRAIN_ROWS,
    'mal_train_rows': MAL_TRAIN_ROWS,
    'valid_rows': VALID_ROWS,
    'test_rows': TEST_ROWS,
    'epochs': EPOCHS,
    'batch_size': TRAIN_BATCH_SIZE,
    'max_length': MAX_LENGTH,
})

## Chargement des donnees enrichies

On garde NovelForge comme domaine d'evaluation. Le train est enrichi avec un echantillon de `manga_dataset.csv`, plus proche du domaine manga/manhwa que l'anime.

In [ ]:
current = load_original_dataset(PROJECT_DIR / 'data' / 'data.csv')
mal_manga = build_enriched_dataset(PROJECT_DIR, include_original=False, include_manga=True, include_anime=False)

print(f'Dataset courant exploitable : {len(current):,}')
print(f'MAL manga exploitable : {len(mal_manga):,}')

display(summarize_labels(current).head(12))
display(summarize_labels(mal_manga).head(12))

In [ ]:
current_train_full, current_temp = train_test_split(current, test_size=0.30, random_state=RANDOM_STATE, shuffle=True)
current_valid_full, current_test_full = train_test_split(current_temp, test_size=0.50, random_state=RANDOM_STATE, shuffle=True)

current_train = current_train_full.sample(n=min(CURRENT_TRAIN_ROWS, len(current_train_full)), random_state=RANDOM_STATE)
mal_train = mal_manga.sample(n=min(MAL_TRAIN_ROWS, len(mal_manga)), random_state=RANDOM_STATE)
valid_df = current_valid_full.sample(n=min(VALID_ROWS, len(current_valid_full)), random_state=RANDOM_STATE)
test_df = current_test_full.sample(n=min(TEST_ROWS, len(current_test_full)), random_state=RANDOM_STATE)

train_df = pd.concat([current_train, mal_train], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Train courant sample : {len(current_train):,}')
print(f'Train MAL sample : {len(mal_train):,}')
print(f'Train total Transformer : {len(train_df):,}')
print(f'Validation NovelForge : {len(valid_df):,}')
print(f'Test NovelForge : {len(test_df):,}')

In [ ]:
mlb = MultiLabelBinarizer(classes=ENRICHED_GENRE_VOCABULARY)
mlb.fit([ENRICHED_GENRE_VOCABULARY])
labels = list(mlb.classes_)
id2label = {index: label for index, label in enumerate(labels)}
label2id = {label: index for index, label in id2label.items()}

X_train = train_df['synopsis_clean'].fillna('').astype(str).to_numpy()
X_valid = valid_df['synopsis_clean'].fillna('').astype(str).to_numpy()
X_test = test_df['synopsis_clean'].fillna('').astype(str).to_numpy()

y_train = mlb.transform(train_df['genre_labels']).astype('float32')
y_valid = mlb.transform(valid_df['genre_labels']).astype('float32')
y_test = mlb.transform(test_df['genre_labels']).astype('float32')

print(f'Labels enrichis : {len(labels)}')
labels

## Fine-tuning du Transformer

Le modele sauvegarde ses checkpoints dans `models/transformer_enriched_novelforge`. Cette experience ne remplace pas le Transformer minimal du notebook 4.

In [ ]:
output_dir = PROJECT_DIR / 'models' / 'transformer_enriched_novelforge'
config = TransformerConfig(
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
    learning_rate=2e-5,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    epochs=EPOCHS,
    weight_decay=0.01,
    threshold=0.5,
    output_dir=str(output_dir),
    logging_steps=50,
)

transformer = NovelForgeTransformer(
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    config=config,
)

start = time.perf_counter()
training_seconds = time.perf_counter() - start
print(f'Training time: {training_seconds:.1f} seconds')

## Calibration des seuils

Comme pour la baseline enrichie, on ne se contente pas du seuil `0.5`. On cherche un seuil global puis des seuils par label sur la validation.

In [ ]:
valid_proba = transformer.predict_proba(X_valid, batch_size=EVAL_BATCH_SIZE)
test_proba = transformer.predict_proba(X_test, batch_size=EVAL_BATCH_SIZE)

best_global_threshold, valid_f1_micro = find_best_global_threshold(y_valid, valid_proba)
label_thresholds = find_best_label_thresholds(y_valid, valid_proba)

print(f'Best global threshold: {best_global_threshold:.2f}')
print(f'Validation F1 micro at best global threshold: {valid_f1_micro:.4f}')
print('First label thresholds:', dict(zip(labels[:8], label_thresholds[:8])))

In [ ]:
def evaluate_probabilities(y_true, probabilities, thresholds):
    y_pred = apply_thresholds(probabilities, thresholds)
    return {
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'jaccard_samples': jaccard_score(y_true, y_pred, average='samples', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'classification_report_text': classification_report(y_true, y_pred, target_names=labels, zero_division=0),
        'classification_report': classification_report(y_true, y_pred, target_names=labels, zero_division=0, output_dict=True),
    }

metrics_global = evaluate_probabilities(y_test, test_proba, best_global_threshold)
metrics_per_label = evaluate_probabilities(y_test, test_proba, label_thresholds)

summary = pd.DataFrame([
    {
        'model': 'transformer_enriched',
        'threshold_strategy': 'global',
        'train_rows': len(train_df),
        'current_train_rows': len(current_train),
        'mal_train_rows': len(mal_train),
        'valid_rows': len(valid_df),
        'test_rows': len(test_df),
        'threshold': best_global_threshold,
        'valid_f1_micro': valid_f1_micro,
        'training_seconds': training_seconds,
        **{key: metrics_global[key] for key in ['f1_micro', 'f1_macro', 'f1_weighted', 'jaccard_samples', 'hamming_loss']},
    },
    {
        'model': 'transformer_enriched',
        'threshold_strategy': 'per_label',
        'train_rows': len(train_df),
        'current_train_rows': len(current_train),
        'mal_train_rows': len(mal_train),
        'valid_rows': len(valid_df),
        'test_rows': len(test_df),
        'threshold': None,
        'valid_f1_micro': None,
        'training_seconds': training_seconds,
        **{key: metrics_per_label[key] for key in ['f1_micro', 'f1_macro', 'f1_weighted', 'jaccard_samples', 'hamming_loss']},
    },
])
summary

In [ ]:
report = pd.DataFrame(metrics_per_label['classification_report']).T
report.loc[labels, ['precision', 'recall', 'f1-score', 'support']].sort_values('f1-score', ascending=False)

## Sauvegarde

Ces artefacts sont separes de ceux du notebook 4. Streamlit detecte automatiquement ce Transformer enrichi s'il existe.

In [ ]:
models_dir = PROJECT_DIR / 'models'
reports_dir = PROJECT_DIR / 'reports'
models_dir.mkdir(exist_ok=True)
reports_dir.mkdir(exist_ok=True)

transformer.save(output_dir)
joblib.dump(labels, models_dir / 'transformer_enriched_labels.joblib')
joblib.dump(label_thresholds, models_dir / 'transformer_enriched_thresholds.joblib')
joblib.dump(
    {
        'summary': summary,
        'labels': labels,
        'global_threshold': best_global_threshold,
        'label_thresholds': label_thresholds,
        'metrics_global': metrics_global,
        'metrics_per_label': metrics_per_label,
        'config': config.__dict__,
    },
    models_dir / 'transformer_enriched_metrics.joblib',
)
summary.to_csv(reports_dir / 'transformer_enriched_metrics.csv', index=False)

print(f'Transformer enrichi sauvegarde : {output_dir}')
print('Labels/seuils/metriques sauvegardes dans models/')
print('Resume sauvegarde dans reports/transformer_enriched_metrics.csv')

## Conclusion

Ce notebook teste l'hypothese du cours dans de meilleures conditions que le notebook 4 : plus de donnees, taxonomie regroupee et calibration des seuils. Le run courant utilise **5 000** lignes d'entrainement, **800** lignes de validation et **1 200** lignes de test NovelForge.

Avec seuil global, il obtient **F1 micro = 0.434** et **F1 macro = 0.209**; avec seuils par label, il obtient **F1 micro = 0.395** et **F1 macro = 0.266**. Le Transformer enrichi progresse donc nettement par rapport au Transformer de demonstration, mais il ne depasse pas la baseline enrichie du notebook 5 (**F1 micro = 0.538**, **F1 macro = 0.491** avec seuils par label).

La conclusion est claire : le Transformer est plus puissant en theorie, mais son avantage pratique depend fortement du volume de donnees, du temps d'entrainement et des ressources de calcul disponibles. Sur CPU, la meilleure solution empirique du projet reste la baseline enrichie.